In [1]:
import geant4_pybind as g4
import numpy as np

In [2]:
class Detector(g4.G4VUserDetectorConstruction):
    def __init__(self, R=2.0 * g4.m, N=40, r=0.1 * g4.m, a=0.5,
                 half_thickness=0.01 * g4.m, rng=None):
        super().__init__()
        self.R, self.N, self.r, self.a = R, N, r, a
        self.h = half_thickness
        self.abs_length = 8.0 * g4.m
        self.light_yield = 10000.0 / g4.MeV
        self.decay_time = 10.0 * g4.ns
        self.spectrum_mean = 3.0 * g4.eV
        self.spectrum_sigma = 0.25 * g4.eV
        self.rng = np.random.default_rng() if rng is None else rng
        if not isinstance(N, (int, np.integer)) or N < 1:
            raise ValueError("N должно быть положительным целым числом.")
        if not (0 < r < R and self.h > 0 and a >= 0):
            raise ValueError("Требуются 0 < r < R, half_thickness > 0, a >= 0.")

        # Весь цилиндр должен лежать внутри сферы, включая внешний обод.
        self.gap = 1.0 * g4.um
        self.center_radius = np.sqrt(R**2 - r**2) - self.h - self.gap
        self.front_radius = self.center_radius - self.h
        if self.front_radius <= 0:
            raise ValueError("Толщина ФЭУ слишком велика для этого детектора.")

        i = np.arange(N)
        z = 1 - 2 * (i + 0.5) / N
        phi = i * np.pi * (3 - np.sqrt(5))
        rho = np.sqrt(1 - z**2)
        self.axes = np.column_stack((rho * np.cos(phi), rho * np.sin(phi), z))
        self.fmt_positions = self.center_radius * self.axes

        distances = np.linalg.norm(
            self.fmt_positions[:, None, :] - self.fmt_positions[None, :, :], axis=2
        )
        np.fill_diagonal(distances, np.inf)
        self.min_distance = float(distances.min())
        # Непересечение описанных вокруг цилиндров шаров — достаточное условие.
        if self.min_distance <= 2 * np.hypot(r, self.h):
            raise ValueError("ФЭУ расположены слишком близко. Уменьшите N или r.")

    def generate_fe(self):
        """Фотоэлектроны от одного попадания: свойство фотокатода детектора."""
        return int(self.rng.poisson(self.a))

    def Construct(self):
        nist = g4.G4NistManager.Instance()
        energies = g4.G4doubleVector([2.0 * g4.eV, 4.0 * g4.eV])

        scint = g4.G4Material("Pseudocumene", 0.876 * g4.g / g4.cm3, 2)
        scint.AddElement(nist.FindOrBuildElement("C"), 9)
        scint.AddElement(nist.FindOrBuildElement("H"), 12)
        scint_mpt = g4.G4MaterialPropertiesTable()
        scint_mpt.AddProperty("RINDEX", energies, g4.G4doubleVector([1.5, 1.5]))
        scint_mpt.AddProperty(
            "ABSLENGTH", energies, g4.G4doubleVector([self.abs_length] * 2)
        )
        # Ширина спектра в условии не задана: оставляем 0.25 эВ из исходного кода.
        spectrum_energy = np.linspace(2.0, 4.0, 201) * g4.eV
        spectrum = np.exp(-0.5 * ((spectrum_energy - self.spectrum_mean) / self.spectrum_sigma)**2)
        scint_mpt.AddProperty(
            "SCINTILLATIONCOMPONENT1",
            g4.G4doubleVector(spectrum_energy.tolist()),
            g4.G4doubleVector(spectrum.tolist()),
        )
        scint_mpt.AddConstProperty("SCINTILLATIONYIELD", self.light_yield)
        scint_mpt.AddConstProperty("SCINTILLATIONTIMECONSTANT1", self.decay_time)
        scint_mpt.AddConstProperty("SCINTILLATIONYIELD1", 1.0)
        scint_mpt.AddConstProperty("RESOLUTIONSCALE", 1.0)
        scint.SetMaterialPropertiesTable(scint_mpt)

        glass = nist.FindOrBuildMaterial("G4_Pyrex_Glass")
        glass_mpt = g4.G4MaterialPropertiesTable()
        # Согласуем показатели, чтобы не отражать фотон перед регистрацией.
        glass_mpt.AddProperty("RINDEX", energies, g4.G4doubleVector([1.5, 1.5]))
        glass.SetMaterialPropertiesTable(glass_mpt)

        air = nist.FindOrBuildMaterial("G4_AIR")
        air_mpt = g4.G4MaterialPropertiesTable()
        air_mpt.AddProperty("RINDEX", energies, g4.G4doubleVector([1.0, 1.0]))
        air.SetMaterialPropertiesTable(air_mpt)

        world_solid = g4.G4Box("World", 1.5 * self.R, 1.5 * self.R, 1.5 * self.R)
        self.world_log = g4.G4LogicalVolume(world_solid, air, "World")
        self.world_phys = g4.G4PVPlacement(
            None, g4.G4ThreeVector(), self.world_log, "World", None, False, 0
        )

        sphere = g4.G4Orb("Sphere", self.R)
        self.scint_log = g4.G4LogicalVolume(sphere, scint, "Sphere")
        self.scint_phys = g4.G4PVPlacement(
            None, g4.G4ThreeVector(), self.scint_log, "Sphere", self.world_log, False, 0
        )

        # Как в задаче 3, внешняя граница поглощает фотоны, не возвращая их назад.
        self.black_surface = g4.G4OpticalSurface(
            "AbsorbingBoundary", g4.G4OpticalSurfaceModel.unified,
            g4.G4OpticalSurfaceFinish.polished, g4.G4SurfaceType.dielectric_metal
        )
        black_mpt = g4.G4MaterialPropertiesTable()
        black_mpt.AddProperty("REFLECTIVITY", energies, g4.G4doubleVector([0.0, 0.0]))
        black_mpt.AddProperty("EFFICIENCY", energies, g4.G4doubleVector([0.0, 0.0]))
        self.black_surface.SetMaterialPropertiesTable(black_mpt)
        self.black_border = g4.G4LogicalBorderSurface(
            "SphereToWorld", self.scint_phys, self.world_phys, self.black_surface
        )

        fmt_solid = g4.G4Tubs("FMT", 0, self.r, self.h, 0, 360 * g4.deg)
        self.fmt_log = g4.G4LogicalVolume(fmt_solid, glass, "FMT")
        self.fmt_volumes = []
        for k, (axis, position) in enumerate(zip(self.axes, self.fmt_positions)):
            # Активный поворот локальной оси z в радиальное направление.
            rotation = g4.G4RotationMatrix()
            rotation.rotateY(float(np.arccos(axis[2])))
            rotation.rotateZ(float(np.arctan2(axis[1], axis[0])))
            transform = g4.G4Transform3D(
                rotation, g4.G4ThreeVector(*map(float, position))
            )
            volume = g4.G4PVPlacement(
                transform, self.fmt_log, f"FMT_{k}", self.scint_log, False, k
            )
            self.fmt_volumes.append(volume)

        self.geometry_ok = not self.scint_phys.CheckOverlaps(2000, 0.0, False)
        for volume in self.fmt_volumes:
            if volume.CheckOverlaps(2000, 0.0, False):
                self.geometry_ok = False
        if not self.geometry_ok:
            raise RuntimeError("Geant4 обнаружил пересечение объемов.")
        return self.world_phys

In [3]:
class ElectronGenerator(g4.G4VUserPrimaryGeneratorAction):
    def __init__(self, energy=2.0 * g4.MeV, rng=None):
        super().__init__()
        self.rng = np.random.default_rng() if rng is None else rng
        self.energy = energy
        self.gun = g4.G4ParticleGun(1)
        self.gun.SetParticleDefinition(g4.G4Electron.ElectronDefinition())
        self.gun.SetParticleEnergy(energy)
        self.gun.SetParticlePosition(g4.G4ThreeVector(0, 0, 0))

    def GeneratePrimaries(self, event):
        cos_theta = self.rng.uniform(-1, 1)
        phi = self.rng.uniform(0, 2 * np.pi)
        sin_theta = np.sqrt(1 - cos_theta**2)
        direction = g4.G4ThreeVector(
            float(sin_theta * np.cos(phi)),
            float(sin_theta * np.sin(phi)),
            float(cos_theta),
        )
        self.gun.SetParticleMomentumDirection(direction)
        self.gun.GeneratePrimaryVertex(event)

In [4]:
def optical_origin(track):
    """Индекс механизма рождения оптического фотона."""
    process = track.GetCreatorProcess()
    name = process.GetProcessName() if process is not None else ""
    if name == "Scintillation":
        return 0
    if name == "Cerenkov":
        return 1
    return 2


class Stepping(g4.G4UserSteppingAction):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector
        self.reset()

    def reset(self):
        self.generated = np.zeros(3, dtype=int)
        self.hits = np.zeros((3, self.detector.N), dtype=int)
        self.photoelectrons = np.zeros((3, self.detector.N), dtype=int)
        self.bulk_absorbed = 0
        self.boundary_absorbed = 0
        self.escaped = 0
        self.edep = 0.0
        self.scint_energy_sum = 0.0
        self.scint_energy2_sum = 0.0
        self.scint_time_sum = 0.0

    def UserSteppingAction(self, step):
        track = step.GetTrack()
        pre_touchable = step.GetPreStepPoint().GetTouchable()
        pre = pre_touchable.GetVolume() if pre_touchable is not None else None

        if track.GetDefinition().GetParticleName() != "opticalphoton":
            if pre == self.detector.scint_phys:
                self.edep += step.GetTotalEnergyDeposit()
            return
        post_point = step.GetPostStepPoint()
        post_touchable = post_point.GetTouchable()
        post = post_touchable.GetVolume() if post_touchable is not None else None

        if post is not None and post.GetLogicalVolume() == self.detector.fmt_log:
            origin = optical_origin(track)
            fmt_id = post.GetCopyNo()
            self.hits[origin, fmt_id] += 1
            self.photoelectrons[origin, fmt_id] += self.detector.generate_fe()
            track.SetTrackStatus(g4.G4TrackStatus.fStopAndKill)
            return
        if pre == self.detector.scint_phys and post == self.detector.world_phys:
            self.boundary_absorbed += 1
            track.SetTrackStatus(g4.G4TrackStatus.fStopAndKill)
            return
        if post is None:
            self.escaped += 1
            return
        process = post_point.GetProcessDefinedStep()
        if process is not None and process.GetProcessName() == "OpAbsorption":
            self.bulk_absorbed += 1


class Tracking(g4.G4UserTrackingAction):
    def __init__(self, state):
        super().__init__()
        self.state = state

    def PreUserTrackingAction(self, track):
        if track.GetDefinition().GetParticleName() != "opticalphoton":
            return
        origin = optical_origin(track)
        self.state.generated[origin] += 1
        if origin == 0:
            energy = track.GetKineticEnergy()
            self.state.scint_energy_sum += energy
            self.state.scint_energy2_sum += energy**2
            self.state.scint_time_sum += track.GetGlobalTime()

In [5]:
def central_hit_probability(detector):
    """Вероятность регистрации фотона от изотропного источника в центре."""
    d = detector.front_radius
    mu_min = d / np.hypot(d, detector.r)
    nodes, weights = np.polynomial.legendre.leggauss(64)
    mu = (1 + mu_min) / 2 + (1 - mu_min) * nodes / 2
    survival = np.exp(-d / (detector.abs_length * mu))
    return detector.N / 2 * (1 - mu_min) / 2 * np.dot(weights, survival)


def sample_summary(values):
    """Среднее, выборочное СКО событий и стандартная ошибка среднего."""
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    std = float(values.std(ddof=1))
    sem = std / np.sqrt(len(values))
    return mean, std, sem

In [6]:

seed = 42
N_events = 20
electron_energy = 2.0 * g4.MeV
if N_events < 2:
    raise ValueError("Для оценки разброса и ошибки среднего нужны хотя бы 2 события.")
source_seed, response_seed = np.random.SeedSequence(seed).spawn(2)
g4.G4Random.setTheSeed(seed)

run_mgr = g4.G4RunManager()
detector = Detector(rng=np.random.default_rng(response_seed))
run_mgr.SetUserInitialization(detector)
physics = g4.FTFP_BERT()
physics.SetVerboseLevel(0)
physics.RegisterPhysics(g4.G4OpticalPhysics())
run_mgr.SetUserInitialization(physics)
g4.G4EmParameters.Instance().SetVerbose(0)
g4.G4HadronicParameters.Instance().SetVerboseLevel(0)

generator = ElectronGenerator(electron_energy, np.random.default_rng(source_seed))
stepping = Stepping(detector)
tracking = Tracking(stepping)
run_mgr.SetUserAction(generator)
run_mgr.SetUserAction(stepping)
run_mgr.SetUserAction(tracking)
run_mgr.Initialize()

optical_manager = g4.G4OpticalPhoton.OpticalPhotonDefinition().GetProcessManager()
for process in optical_manager.GetProcessList():
    if process.GetProcessName() == "Scintillation":
        optical_manager.SetProcessActivation(process, False)

g4.G4Random.setTheSeed(seed)

records = []
print("\nСобытие  E_dep, МэВ  N_scint  N_Cherenkov  Попадания  Q_scint  Q_Cherenkov  Q_total")
for event_id in range(N_events):
    stepping.reset()
    run_mgr.BeamOn(1)  # Теперь одно событие Geant4 — один электрон 2 МэВ.
    hits = stepping.hits.sum(axis=1)
    charge = stepping.photoelectrons.sum(axis=1)
    generated = stepping.generated.copy()
    accounted = int(hits.sum()) + stepping.bulk_absorbed + stepping.boundary_absorbed + stepping.escaped
    if accounted != int(generated.sum()):
        raise RuntimeError(f"Не сошелся баланс фотонов в событии {event_id + 1}.")
    record = {
        "event": event_id + 1,
        "edep_MeV": stepping.edep / g4.MeV,
        "n_scint": int(generated[0]),
        "n_cherenkov": int(generated[1]),
        "n_other": int(generated[2]),
        "hits": int(hits.sum()),
        "q_scint": int(charge[0]),
        "q_cherenkov": int(charge[1]),
        "q_other": int(charge[2]),
        "q_total": int(charge.sum()),
        "accounted": accounted,
        "scint_energy_sum": stepping.scint_energy_sum,
        "scint_energy2_sum": stepping.scint_energy2_sum,
        "scint_time_sum": stepping.scint_time_sum,
    }
    records.append(record)
    print(f"{event_id + 1:7d}  {record['edep_MeV']:10.6f}  {generated[0]:7d}  {generated[1]:11d}  "
          f"{record['hits']:9d}  {charge[0]:7d}  {charge[1]:11d}  {record['q_total']:7d}")

charges = np.array([r['q_total'] for r in records])
scint_charges = np.array([r['q_scint'] for r in records])
cherenkov_charges = np.array([r['q_cherenkov'] for r in records])
mean_Q, std_Q, sem_Q = sample_summary(charges)
mean_scint, std_scint, sem_scint = sample_summary(scint_charges)
mean_cherenkov, std_cherenkov, sem_cherenkov = sample_summary(cherenkov_charges)
mean_edep = np.mean([r['edep_MeV'] for r in records]) * g4.MeV
mean_n_scint = np.mean([r['n_scint'] for r in records])
mean_n_cherenkov = np.mean([r['n_cherenkov'] for r in records])

p_hit = central_hit_probability(detector)
n_nominal = detector.light_yield * electron_energy
p_surface = detector.N * detector.r**2 / (4 * detector.R**2)
Q_no_absorption = detector.a * n_nominal * p_surface
Q_surface = Q_no_absorption * np.exp(-detector.R / detector.abs_length)
Q_cylinders = detector.a * n_nominal * p_hit
Q_scint_from_edep = detector.a * detector.light_yield * mean_edep * p_hit
Q_total_estimate = Q_scint_from_edep + detector.a * mean_n_cherenkov * p_hit

n_scint_all = sum(r['n_scint'] for r in records)
mean_photon_energy = sum(r['scint_energy_sum'] for r in records) / n_scint_all
variance_photon_energy = sum(r['scint_energy2_sum'] for r in records) / n_scint_all - mean_photon_energy**2
std_photon_energy = np.sqrt(max(0.0, variance_photon_energy))
mean_emission_time = sum(r['scint_time_sum'] for r in records) / n_scint_all

print("\nСРАВНЕНИЕ С ОЖИДАНИЕМ")
print(f"Geant4: {g4.G4Version}; seed = {seed}; событий = {N_events}")
print(f"Перекрытий нет: {detector.geometry_ok}; баланс фотонов выполнен во всех событиях.")
print(f"Среднее энерговыделение в сцинтилляторе: {mean_edep / g4.MeV:.6f} МэВ")
print(f"Сцинтилляционных фотонов на событие: {mean_n_scint:.2f}; Y * <E_dep> = {detector.light_yield * mean_edep:.2f}")
print(f"Черенковских фотонов на событие: {mean_n_cherenkov:.2f}")
print(f"Средняя энергия сцинтилляционных фотонов: {mean_photon_energy / g4.eV:.4f} эВ")
print(f"Ширина спектра (СКО): {std_photon_energy / g4.eV:.4f} эВ")
print(f"Среднее время рождения сцинтилляционных фотонов от начала события: {mean_emission_time / g4.ns:.4f} нс")
print(f"Оценка без поглощения, 2 МэВ: {Q_no_absorption:.2f} фотоэлектронов")
print(f"Оценка с поглощением и ФЭУ на сфере: {Q_surface:.2f}")
print(f"Сцинтилляционный заряд для цилиндров, точечный центр: {Q_cylinders:.2f}")
print(f"Та же оценка по фактическому <E_dep>: {Q_scint_from_edep:.2f}")
print(f"Сцинтилляционный заряд Geant4: {mean_scint:.2f} +/- {sem_scint:.2f} (ошибка среднего)")
if sem_scint > 0:
    print(f"Отклонение среднего сцинтилляционного заряда от центральной оценки: {(mean_scint - Q_scint_from_edep) / sem_scint:.2f} ошибки среднего")
print(f"Средний черенковский вклад: {mean_cherenkov:.2f} +/- {sem_cherenkov:.2f}")
print(f"Полный заряд Geant4: {mean_Q:.2f} +/- {sem_Q:.2f} (ошибка среднего)")
print(f"СКО заряда отдельного события: {std_Q:.2f}")
print(f"Ориентир для полного среднего с учетом числа черенковских фотонов: {Q_total_estimate:.2f}")


**************************************************************
 Geant4 version Name: geant4-11-04-patch-01 [MT]   (13-March-2026)
                       Copyright : Geant4 Collaboration
                      References : NIM A 506 (2003), 250-303
                                 : IEEE-TNS 53 (2006), 270-278
                                 : NIM A 835 (2016), 186-225
                             WWW : http://geant4.org/
**************************************************************

<<< Geant4 Physics List simulation engine: FTFP_BERT

### Adding tracking cuts for neutron  TimeCut(ns)= 10000  KinEnergyCut(MeV)= 0
### Birks coefficients used in run time

Событие  E_dep, МэВ  N_scint  N_Cherenkov  Попадания  Q_scint  Q_Cherenkov  Q_total
      1    2.000000    20208          426        407      227            2      229
      2    2.000000    20153          457        414      206            7      213
      3    2.000000    20004          420        372      206            3      209


In [7]:
from IPython.display import Markdown, display

display(Markdown("\n".join([
    "| Величина | Результат |",
    "|---|---:|",
    f"| Число электронов | {N_events} |",
    f"| Среднее энерговыделение, МэВ | {mean_edep / g4.MeV:.6f} |",
    f"| Сцинтилляционных фотонов на событие | {mean_n_scint:.2f} |",
    f"| Черенковских фотонов на событие | {mean_n_cherenkov:.2f} |",
    f"| Сцинтилляционный заряд, фотоэлектроны | {mean_scint:.2f} ± {sem_scint:.2f} |",
    f"| Черенковский вклад, фотоэлектроны | {mean_cherenkov:.2f} ± {sem_cherenkov:.2f} |",
    f"| Полный заряд, фотоэлектроны | {mean_Q:.2f} ± {sem_Q:.2f} |",
    f"| СКО полного заряда отдельного события | {std_Q:.2f} |",
    f"| Центральная оценка сцинтилляционного заряда | {Q_scint_from_edep:.2f} |",
])))
if sem_scint > 0:
    print(f"Разность среднего сцинтилляционного заряда и оценки: {(mean_scint - Q_scint_from_edep) / sem_scint:.2f} ошибки среднего.")
print("Средние для 1 и 2 МэВ нужно сравнивать при одинаковых поглощении, геометрии и наборе оптических процессов.")

| Величина | Результат |
|---|---:|
| Число электронов | 20 |
| Среднее энерговыделение, МэВ | 2.000000 |
| Сцинтилляционных фотонов на событие | 20035.00 |
| Черенковских фотонов на событие | 384.45 |
| Сцинтилляционный заряд, фотоэлектроны | 202.30 ± 4.11 |
| Черенковский вклад, фотоэлектроны | 5.00 ± 0.74 |
| Полный заряд, фотоэлектроны | 207.30 ± 3.98 |
| СКО полного заряда отдельного события | 17.78 |
| Центральная оценка сцинтилляционного заряда | 199.30 |

Разность среднего сцинтилляционного заряда и оценки: 0.73 ошибки среднего.
Средние для 1 и 2 МэВ нужно сравнивать при одинаковых поглощении, геометрии и наборе оптических процессов.
